In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pillow huggingface_hub sentencepiece

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("huggingface")
login(token=hf_token)

In [ ]:
model_id = "google/paligemma-3b-mix-224"

processor = AutoProcessor.from_pretrained(model_id, token=hf_token)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token
).eval()

preprocessor_config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
import os
import json

In [ ]:
base_dir = "/content/drive/MyDrive/프젝랩/output"
md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드.md")
new_md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_보충.md")
image_dir = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_images")

In [ ]:
def select_image(image_path, jpg=False):
  img = Image.open(image_path)
  width, height = img.size

  if width < 100 and height < 100:
    return False, "해상도 낮음"

  if jpg:
    return True, "jpg 파일"

  file_size = os.path.getsize(image_path)/ 1024
  if file_size < 8:
    return False, "파일 크기 작음"

  img_rgb = img.convert("RGB")
  colors = img_rgb.getcolors(maxcolors=5000)
  if colors is not None and len(colors) < 150:
    return False, "단순 그래픽"
  return True, "유의미한 그래픽"


In [ ]:
with open(md_path, "r", encoding="utf-8") as f:
  md_lines = f.readlines()

tasks = []
if os.path.exists(image_dir):
  for f in os.listdir(image_dir):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
      tasks.append((os.path.join(image_dir, f), False, f))

In [ ]:
processed = 0
image_descriptions = {}

for full_path, is_jpg, filename in tasks:
  passed, reason = select_image(full_path, is_jpg)
  if not passed:
    continue

  image = Image.open(full_path).convert("RGB")
  prompt = "<image>\ncaption ko: 산업안전 가이드라인 이미지에 등장하는 구체적인 행동, 위험요소, 작업자, 장비, 표지, 한국어 표, 그래프의 맥락을 상세히 설명해줘."

  inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

  input_len = inputs["input_ids"].shape[-1]

  with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=250, do_sample=True, temperature=0.1, repetition_penalty=1.3)

  result_text = processor.decode(output[0], skip_special_tokens=True).strip()

  prefix = "caption ko:"
  if prefix in result_text:
    result_text = result_text.split(prefix, 1)[1].strip()

  clean_text = result_text.replace(prompt.replace("<image>\n", ""), "").strip()

  print("="*10)
  print("filename:", filename)
  print("description:")
  print(clean_text)

  processed += 1
  image_descriptions[filename] = clean_text

result_path = "/content/drive/MyDrive/프젝랩/output/image_descriptions.json"
with open(result_path, "w", encoding="utf-8") as f:
  json.dump(image_descriptions, f, ensure_ascii=False, indent=4)


filename: imageFile2.png
description:
산업안전 가이드라인 이미지에 등장하는 구체적인 행동, 위험요소, 작업자, 장비, 표지, 한국어 표, 그래프의 맥락을 상세히 설명해줘.
이미지는 산업 안전 가이드라인에서 등장 하는 구체적인 행동, 위험요소, 작업자, 장비, 표지 및 그래프를 보여줍니다. 이는 각 항목에 대한 자세한 정보와 함께 사용할 수 있는 유용한 도구가 될 것입니다.


KeyboardInterrupt: 

In [ ]:
with open(result_path, "r", encoding="utf-8") as f:
  descriptions = json.load(f)

with open(md_path, "r", encoding="utf-8") as f:
  md_lines = f.readlines()

description_lines = []
for line in md_lines:


gemma 3 사용시

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pillow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 27.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("huggingface")
login(token=token)

model_id = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(model_id, token=token)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=token
).eval()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
!nvidia-smi

Wed May 20 17:33:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             29W /   70W |    3269MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
processed = 0
image_descriptions = {}

for full_path, is_jpg, filename in tasks:
  passed, reason = select_image(full_path, is_jpg)
  if not passed:
    continue

  image = Image.open(full_path).convert("RGB")
  prompt = """
  산업안전 가이드라인 문서에서 추출된 이미지를 보고 한국어로 설명해.

  1. 무의미 판정
  장식용 아이콘, 단순 구분선, 레이아웃 조각, 의미없는 단순한 그림 등 정보가 거의 없다면,
  반드시 '무의미'만 출력해

  2. 무의미하지 않은 이미지라면, 한국어 줄글 1~2문장으로 설명해.
  보이는 대상(장비, 사람), 작업 상황, 산업안전상 관련 위험요인이나 주의점을 포함해.
  이미지 내 텍스트가 포함되면, 포함하되 글자가 불확실하면 '텍스트 판독 불확실'이라고 답해.
  표나 그래프가 포함되면 2문장 이내로 설명해.

  주의사항:
  이미지에 없는 내용은 추측하지 마.
  무의미라고 판단하면, 추가 설명은 하지마.
  별표, 불필요한 설명 등은 사용하지마.
  """

  messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": image},
              {"type": "text", "text": prompt}
          ]
      }
  ]

  inputs = processor.apply_chat_template(
      messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True
  ).to(model.device)

  input_len = inputs["input_ids"].shape[-1]

  with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)

  generated_tokens = outputs[0][input_len:]
  result_text = processor.decode(generated_tokens, skip_special_tokens=True).strip()
  #result_text = processor.decode(outputs[0], skip_special_tokens=True).strip()

  print("="*10)
  print("filename:", filename)
  print("description:")
  print(result_text)

  processed += 1
  image_descriptions[filename] = result_text

result_path = "/content/drive/MyDrive/프젝랩/output/image_descriptions.json"
with open(result_path, "w", encoding="utf-8") as f:
  json.dump(image_descriptions, f, ensure_ascii=False, indent=4)


filename: imageFile2.png
description:
이미지에 나타난 아이콘들을 분석한 결과는 다음과 같습니다.

1.  **무의미**
    *   장식용 아이콘
    *   단순 구분선
    *   레이아웃 조각
    *   의미 없는 단순한 그림

2.  **무의미하지 않은 이미지**
    *   **사람 아이콘:** 작업자 또는 관련 인력을 나타냅니다. 작업 중 안전 수칙 준수 및 위험 인지의 중요성을 강조합니다.
    *   **돈 아이콘:** 금전적인 손실을 방지하는 안전 관리의 필요성을 나타냅니다.
    *   **사람 묶음 아이콘:** 불필요한 방문이나 출입을 제한하여 안전 사고를 예방하는 것을 의미합니다.
filename: imageFile3.png
description:
**무의미**
filename: imageFile4.png
description:
**무의미**

이미지는 지하철역의 이동 통로로, 사람들의 이동을 돕는 이동식 경사로입니다. 텍스트 판독 불확실
filename: imageFile5.png
description:
무의미
filename: imageFile6.png
description:
무의미
filename: imageFile7.png
description:
무의미
filename: imageFile8.png
description:
이미지에 보이는 것은 위생장비입니다. 이 장비는 간주가 구동기나 유압장에 의해 작동하는 것으로 보입니다. 

**무의미**
filename: imageFile10.png
description:
**무의미**

이미지는 컴퓨터 메인보드와 관련된 장비의 모습입니다. 메인보드는 컴퓨터의 핵심 부품으로, 전원 공급 장치, CPU, 메모리 등 다양한 부품들을 연결하는 역할을 합니다.
filename: imageFile11.png
description:
무의미
filename: imageFile12.png
description:
이미지에 나타난 산업안전 가이드라인 문서의 내용을 분석

KeyboardInterrupt: 

In [ ]:
print(next(model.parameters()).dtype)

torch.float16


qwen 모델 사용시

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers accelerate bitsandbytes pillow huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.3 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.8.0.dev0 which is incompatible.
unsloth 2026.5.5 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.8.0.dev0 which is incompatible.


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import json
import gc
import re
import torch
from PIL import Image

In [ ]:
import os
import json

base_dir = "/content/drive/MyDrive/프젝랩/output"
md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드.md")
new_md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_보충.md")
image_dir = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_images")

In [ ]:
def select_image(image_path, jpg=False):
  img = Image.open(image_path)
  width, height = img.size

  if width < 100 and height < 100:
    return False, "해상도 낮음"

  if jpg:
    return True, "jpg 파일"

  file_size = os.path.getsize(image_path)/ 1024
  if file_size < 8:
    return False, "파일 크기 작음"

  img_rgb = img.convert("RGB")
  colors = img_rgb.getcolors(maxcolors=5000)
  if colors is not None and len(colors) < 150:
    return False, "단순 그래픽"
  return True, "유의미한 그래픽"


In [ ]:
with open(md_path, "r", encoding="utf-8") as f:
  md_lines = f.readlines()

tasks = []
if os.path.exists(image_dir):
  for f in os.listdir(image_dir):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
      tasks.append((os.path.join(image_dir, f), False, f))

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

hf_token = userdata.get("huggingface")
login(hf_token)
os.environ["HF_TOKEN"] = hf_token

In [ ]:
!pip install unsloth unsloth_zoo
!apt-get update
!apt-get install -y poppler-utils
!pip install pdf2image
!pip install decord

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # RoPE scaling internally 지원
dtype = torch.float16
load_in_4bit = True


try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit", # Qwen3 (14B) fit comfortably in a google colab 16gb vram tesla t4 gpu
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = "auto", # If it still causes CUDA out of memory error, try 'cpu' or 'auto' or 'balanced'
        full_finetuning = False,
        token = hf_token
    )
except TimeoutError:
    !pip install modelscope
    import os
    os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit", # Use the original model name
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = "auto", # If it still causes CUDA out of memory error, try 'cpu' or 'auto' or 'balanced'
        full_finetuning = False,
        token = hf_token
    )


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 27.2 MB/s eta 0:00:00
==((====))==  Unsloth 2026.5.5: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfl

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

In [ ]:
import os
import json
import gc
import re
import torch
from PIL import Image

In [ ]:
import os
import json

base_dir = "/content/drive/MyDrive/프젝랩/output"
md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_split.md")
new_md_path = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_split_보충.md")
image_dir = os.path.join(base_dir, "(교재) 엘리베이터 안전작업가이드_split_images")

In [ ]:
def select_image(image_path, jpg=False):
  img = Image.open(image_path)
  width, height = img.size

  if width < 100 and height < 100:
    return False, "해상도 낮음"

  if jpg:
    return True, "jpg 파일"

  file_size = os.path.getsize(image_path)/ 1024
  if file_size < 8:
    return False, "파일 크기 작음"

  img_rgb = img.convert("RGB")
  colors = img_rgb.getcolors(maxcolors=5000)
  if colors is not None and len(colors) < 150:
    return False, "단순 그래픽"
  return True, "유의미한 그래픽"


In [ ]:
with open(md_path, "r", encoding="utf-8") as f:
  md_lines = f.readlines()

tasks = []
if os.path.exists(image_dir):
  for f in os.listdir(image_dir):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
      tasks.append((os.path.join(image_dir, f), False, f))

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit")

processed = 0
image_descriptions = {}

for full_path, is_jpg, filename in tasks:
  passed, reason = select_image(full_path, is_jpg)
  if not passed:
    continue

  image = Image.open(full_path).convert("RGB")
  prompt = """
  산업안전 가이드라인 문서에서 추출된 이미지를 보고 한국어로 설명해.
  출력은 두 가지 중 하나야

  1. 무의미한 이미지인 경우:
  장식용 아이콘, 단순 구분선, 레이아웃 조각, 의미없는 단순한 그림 등 정보가 거의 없다면,
  반드시 '무의미'만 출력.

  2. 유의미한 이미지인 경우:
  한국어 줄글 1~2문장으로 설명.
  다음에 따라 설명:
  - 장비명, 작업상황, 표지 내용 중심
  - 위험요인은 명확히 보이면 포함
  - 이미지 내 텍스트가 포함되면, 포함하되 글자가 불확실하면 '(텍스트 판독 불확실)' 붙임
  - 표나 그래프가 포함되면 2문장 이내로 설명

  주의사항:
  이미지에 없는 내용은 추측하지 마.
  무의미라고 판단하면, 추가 설명은 하지마.
  별표, 불필요한 설명 등은 사용하지마.
  """

  messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": image},
              {"type": "text", "text": prompt}
          ]
      }
  ]

  gc.collect()
  torch.cuda.empty_cache()

  inputs = processor.apply_chat_template(
      messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True
  ).to(model.device)

  input_len = inputs["input_ids"].shape[-1]

  with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=60, do_sample=False, repetition_penalty=1.15)

  generated_tokens = outputs[0][input_len:]
  result_text = processor.decode(generated_tokens, skip_special_tokens=True).strip()
  #result_text = processor.decode(outputs[0], skip_special_tokens=True).strip()

  print("="*10)
  print("filename:", filename)
  print("description:")
  print(result_text)

  processed += 1
  image_descriptions[filename] = result_text

result_path = "/content/drive/MyDrive/프젝랩/output/image_descriptions.json"
with open(result_path, "w", encoding="utf-8") as f:
  json.dump(image_descriptions, f, ensure_ascii=False, indent=4)


filename: imageFile59.png
description:
무의미
filename: imageFile7.png
description:
엘리베이터 유압식 시스템의 내부 구조를 보여주는 도면이다. 유압식이라는 특징을 통해 작동 원리를 나타내며, 상부에는 제어 및 안전 장치가 설치되어 있는 것으로 보인다.
filename: imageFile30.png
description:
승강기 내부 구성도로, 전동기, 캐빈, 안전장치 및 운행 시스템을 보여주는 기술 도면이다.
filename: imageFile67.png
description:
산업용 기계 부품의 해체 또는 점검 현장. 회전하는 바퀴와 연결된 메커니즘 부분을 보여주며, 고정 장치 및 연결부가 노출되어 있어 작업 시 인력 사고 위험이 존재함.
filename: imageFile17.png
description:
유압계에 의해 운반되는 물품을 위한 슬라이딩 플랫폼형 웨이크업 기기(‘월체어프트’)이며, 고정된 위치에서 물품을 안전하게 들어올리는 작업 상황이다.
filename: imageFile19.png
description:
리미트스위치류 장비 설치 모습. 벽면에 고정되어 있는 전자기기와 연결된 스위치 구성 요소를 보여주며, 안전 장치 작동 시 신호를 전달하는 기능을 수행한다.
filename: imageFile57.png
description:
산업용 문 또는 층간 분리 장치의 일부로, 좌우로 나뉘어 있는 금속 프레임과 연결된 전기 케이블 및 센서(‘XU’, ‘TX’ 표시)를 보여주는 설치
filename: imageFile53.png
description:
산업용 스프링 압축기 또는 기계적 작동 장치로 보이며, 고정된 기반 위에 코일 스프링과 연결된 핸들 및 전자 부품(케이블)이 있는 상태로 구성되어 있다.
filename: imageFile16.png
description:
흰색 급속도어로 된 문이며, 문 안쪽에는 ‘(텍스트 판독 불확실)’이라는 표시와 함께 안전 관련 경고 또는 지침이 붙어 있는

In [ ]:
enriched_lines = []
for line in md_lines:
  enriched_lines.append(line)

  for filename, caption in image_descriptions.items():
    if filename in line and "![image" in line:
      caption_block = f"\n시각 자료 분석\n{caption}\n"
      enriched_lines.append(caption_block)

with open(new_md_path, 'w', encoding='utf-8') as f:
  f.writelines(enriched_lines)